In [ ]:
import numpy as np
import pandas as pd

num_trials = 203
num_units = 131

response = np.where(np.random.rand(num_trials) < 0.51, 1, -1)
rewarded = np.where(np.random.rand(num_trials) < 0.80, 1, 0)
block_side = np.where(np.random.rand(num_trials) < 0.50, 1, -1)
rewarded_prev = np.where(np.random.rand(num_trials) < 0.80, 1, 0)

rands = np.random.rand(num_trials)
response_prev = np.where(rands < 0.45, -1, 1)
response_prev[np.where((response_prev == 1) & (rands < 0.55))[0]] = 0

tvs = pd.DataFrame(
    {
        "response": response,
        "rewarded": rewarded,
        "block_side": block_side,
        "response_prev": response_prev,
        "rewarded_prev": rewarded_prev,
    }
)

In [ ]:
mn = -2
mx = 2

mean = (mn + mx) / 2
std = (mx - mn) / 6  # ~99.7% of values fall within [mn, mx]


def gen_trace():
    weights = np.random.normal(mean, std, size=(1, 5))
    trace = np.sum(weights @ np.array(tvs).T, axis=0) + np.random.randn(num_trials)

    return weights.flatten(), trace


out = [gen_trace() for _ in range(num_units)]
weights = np.array([out[i][0] for i in range(num_units)])
robs = np.array([out[i][1] for i in range(num_units)]).T

In [ ]:
from sklearn.linear_model import RidgeCV

enc = RidgeCV(
    alphas=np.logspace(-5, 5, 11, base=10),
    alpha_per_target=True,
).fit(tvs, robs)
enc.coef_

In [ ]:
np.all(np.isclose(enc.coef_, weights))

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.imshow(enc.coef_ - weights, interpolation="none", aspect="auto")
plt.colorbar()
plt.show()